In [66]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

In [67]:

# Funzione per pulire numeri sporchi (es. "1,200.50 $")
def clean_currency(x):
    if isinstance(x, str):
        # Rimuove virgole, dollari, percentuali e spazi
        clean_str = x.replace(',', '').replace('$', '').replace('%', '').strip()
        if clean_str == '': return np.nan
        return float(clean_str)
    return x

# Carichiamo i file
print("Caricamento dataset...")
comm = pd.read_csv(os.path.join('communications_data.csv'))
demo = pd.read_csv(os.path.join('demographics_data.csv'))
econ = pd.read_csv(os.path.join('economy_data.csv'))
energy = pd.read_csv(os.path.join('energy_data.csv'))
geog = pd.read_csv(os.path.join('geography_data.csv'))
gov = pd.read_csv(os.path.join('government_and_civics_data.csv'))
trans = pd.read_csv(os.path.join('transportation_data.csv'))

Caricamento dataset...


In [68]:
df = demo.merge(comm, on='Country', how='left') \
         .merge(demo, on='Country', how='left') \
         .merge(econ, on='Country', how='left') \
         .merge(energy, on='Country', how='left')\
         .merge(geog, on='Country', how='left') \
         .merge(trans, on='Country', how='left')\
         .merge(gov, on='Country', how='left') 



In [69]:
df.head()

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 259 entries, 0 to 258
Data columns (total 85 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country                                            259 non-null    object 
 1   Total_Population_x                                 237 non-null    object 
 2   Population_Growth_Rate_x                           237 non-null    object 
 3   Birth_Rate_x                                       228 non-null    float64
 4   Death_Rate_x                                       230 non-null    float64
 5   Net_Migration_Rate_x                               229 non-null    float64
 6   Median_Age_x                                       227 non-null    float64
 7   Sex_Ratio_x                                        227 non-null    float64
 8   Infant_Mortality_Rate_x                            227 non-null    float64
 9   Total_Fert

In [70]:
cols_to_clean = [
    'mobile_cellular_subscriptions_total', 'internet_users_total', 'electricity_access_percent', 'Total_Population_x',
    'Male_Literacy_Rate_x', 'Female_Literacy_Rate_x', 
    'Youth_Unemployment_Rate_y', 'Population_Below_Poverty_Line_percent',
    'Population_Growth_Rate_y', 'Land_Area',
    'Real_GDP_per_Capita_USD', 'Unemployment_Rate_percent',
    'Government_Type', 'Suffrage_Age', 'Median_Age_y'

]

df_selected = df[cols_to_clean]

In [71]:
df_selected.head()

,mobile_cellular_subscriptions_total,internet_users_total,electricity_access_percent,Total_Population_x,Male_Literacy_Rate_x,Female_Literacy_Rate_x,Youth_Unemployment_Rate_y,Population_Below_Poverty_Line_percent,Population_Growth_Rate_y,Land_Area,Real_GDP_per_Capita_USD,Unemployment_Rate_percent,Government_Type,Suffrage_Age,Median_Age_y
0,23000000.0,7020000,97.0,39232003,39.4%,7.2%,20.2%,54.5,2.26%,"652,230 sq km",1500.0,13.28,Theocracy,18.0,19.5
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2635466.0,2291000,100.0,3101621,38.8%,6%,27.8%,14.3,0.19%,"27,398 sq km",14500.0,11.82,Republic,18.0,34.3
3,47028685.0,31240000,99.0,44758398,41.3%,0.7%,31.9%,5.5,1.27%,"2,381,740 sq km",11000.0,12.70,Republic,18.0,28.9
4,2250.0,18135,59.0,44620,NaN,NaN,NaN,NaN,1.74%,224 sq km,11200.0,29.80,Republic,18.0,27.2


In [72]:
# Selezione colonne per i 4 Pilastri di Rischio

# Pilastro 1: Rischio di Collasso Economico (Economic Resilience)
pillar_1_economic = [
    'Population_Below_Poverty_Line_percent',  # Povertà
    'Public_Debt_percent_of_GDP',              # Debito pubblico
    'Real_GDP_per_Capita_USD'                  # PIL pro capite (va invertito)
]

# Pilastro 2: Rischio di Pressione Sociale e Instabilità (Social Fragility)
pillar_2_social = [
    'Infant_Mortality_Rate_x',       # Mortalità infantile
    'Youth_Unemployment_Rate_y',     # Disoccupazione giovanile
    'Net_Migration_Rate_x'           # Tasso migratorio (va invertito/gestito)
]

# Pilastro 3: Rischio Logistico e Infrastrutturale (Access Constraints)
# Nota: Calcoleremo la densità stradale dalla colonna roadways_km e Land_Area
pillar_3_infrastructure = [
    'electricity_access_percent',  # Accesso all'elettricità
    'roadways_km',                 # Km di strade (per calcolare densità)
    'Land_Area',                   # Area terrestre (per calcolare densità)
    'internet_users_total'         # Utenti Internet totali
]

# Pilastro 4: Rischio Demografico Strutturale (Population Stress)
pillar_4_demographic = [
    'Population_Growth_Rate_y',      # Tasso di crescita della popolazione
    'Median_Age_y',                  # Età mediana (va invertito)
    'Total_Population_y'             # Popolazione totale (per normalizzare connettività)
]

# Combina tutte le colonne + Country per identificare i paesi
all_selected_columns = ['Country'] + pillar_1_economic + pillar_2_social + pillar_3_infrastructure + pillar_4_demographic

# Seleziona dal dataframe merged
df_risk_pillars = df[all_selected_columns].copy()

print(f"Dataset con i 4 pilastri di rischio:")
print(f"Shape: {df_risk_pillars.shape}")
print(f"\nColonne selezionate ({len(all_selected_columns)-1} variabili):")
for i, col in enumerate(all_selected_columns[1:], 1):
    print(f"  {i}. {col}")
print(f"\nPrime 5 righe:")
df_risk_pillars.head()

Dataset con i 4 pilastri di rischio:
Shape: (259, 14)

Colonne selezionate (13 variabili):
  1. Population_Below_Poverty_Line_percent
  2. Public_Debt_percent_of_GDP
  3. Real_GDP_per_Capita_USD
  4. Infant_Mortality_Rate_x
  5. Youth_Unemployment_Rate_y
  6. Net_Migration_Rate_x
  7. electricity_access_percent
  8. roadways_km
  9. Land_Area
  10. internet_users_total
  11. Population_Growth_Rate_y
  12. Median_Age_y
  13. Total_Population_y

Prime 5 righe:


,Country,Population_Below_Poverty_Line_percent,Public_Debt_percent_of_GDP,Real_GDP_per_Capita_USD,Infant_Mortality_Rate_x,Youth_Unemployment_Rate_y,Net_Migration_Rate_x,electricity_access_percent,roadways_km,Land_Area,internet_users_total,Population_Growth_Rate_y,Median_Age_y,Total_Population_y
0,AFGHANISTAN,54.5,7.00,1500.0,103.06,20.2%,0.10,97.0,34903,"652,230 sq km",7020000,2.26%,19.5,39232003
1,AKROTIRI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ALBANIA,14.3,84.06,14500.0,10.54,27.8%,3.22,100.0,3945,"27,398 sq km",2291000,0.19%,34.3,3101621
3,ALGERIA,5.5,27.50,11000.0,19.22,31.9%,0.81,99.0,104000,"2,381,740 sq km",31240000,1.27%,28.9,44758398
4,AMERICAN SAMOA,NaN,12.20,11200.0,9.87,NaN,27.36,59.0,241,224 sq km,18135,1.74%,27.2,44620


In [73]:
df_risk_pillars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 259 entries, 0 to 258
Data columns (total 14 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Country                                259 non-null    object 
 1   Population_Below_Poverty_Line_percent  174 non-null    float64
 2   Public_Debt_percent_of_GDP             203 non-null    float64
 3   Real_GDP_per_Capita_USD                220 non-null    float64
 4   Infant_Mortality_Rate_x                227 non-null    float64
 5   Youth_Unemployment_Rate_y              206 non-null    object 
 6   Net_Migration_Rate_x                   229 non-null    float64
 7   electricity_access_percent             216 non-null    float64
 8   roadways_km                            213 non-null    object 
 9   Land_Area                              247 non-null    object 
 10  internet_users_total                   225 non-null    object 
 11  Popula

In [74]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. FUNZIONI DI UTILITÀ (PULIZIA)
def clean_numeric(x):
    """Converte stringhe con % , $ e virgole in float."""
    if isinstance(x, str):
        # Rimuove virgole, $, %, spazi, unità (sq km, million, etc.)
        clean_str = x.replace(',', '').replace('$', '').replace('%', '').replace('sq km', '').replace('million', '').replace('(percentage)', '').strip()
        if clean_str == '' or clean_str.lower() == 'nan':
            return np.nan
        return float(clean_str)
    return x

# Applichiamo la pulizia a tutte le colonne selezionate (tranne 'Country')
# df_risk_pillars è il dataframe che hai creato nel tuo step precedente
cols_to_clean = [col for col in df_risk_pillars.columns if col != 'Country']
for col in cols_to_clean:
    df_risk_pillars[col] = df_risk_pillars[col].apply(clean_numeric)


In [75]:

# =============================================================================
# 2. CREAZIONE COLONNE DERIVATE
# =============================================================================

# A. Densità Stradale (km strade / Area Terra)
# Evitiamo divisioni per zero
df_risk_pillars['Road_Density'] = df_risk_pillars['roadways_km'] / df_risk_pillars['Land_Area']
df_risk_pillars['Road_Density'] = df_risk_pillars['Road_Density'].replace([np.inf, -np.inf], 0)

# B. Percentuale Utenti Internet (Utenti / Popolazione)
df_risk_pillars['Internet_Penetration'] = df_risk_pillars['internet_users_total'] / df_risk_pillars['Total_Population_y']
df_risk_pillars['Internet_Penetration'] = df_risk_pillars['Internet_Penetration'].replace([np.inf, -np.inf], 0)


In [76]:

# =============================================================================
# 3. ELABORAZIONE DEI 4 PILASTRI (DF SEPARATI)
# =============================================================================

scaler = MinMaxScaler()

def process_pillar(df_source, cols, risk_name, invert_cols=[]):
    """
    1. Seleziona colonne
    2. Elimina NA (specifici per questo pilastro)
    3. Normalizza (0-1)
    4. Inverte le colonne 'positive' (dove alto valore = basso rischio)
    5. Calcola la media (Risk Score)
    """
    # Seleziona Country + colonne specifiche
    temp_df = df_source[['Country'] + cols].copy()
    
    # 3. Normalizzazione
    # Creiamo suffissi _norm per chiarezza
    norm_cols = []
    for col in cols:
        col_name = f"{col}_norm"
        # Reshape necessario per sklearn
        data_values = temp_df[[col]].values
        temp_df[col_name] = scaler.fit_transform(data_values)
        
        # 4. Inversione (Se la colonna è nella lista da invertire)
        # Esempio: GDP alto è bene -> Rischio = 1 - GDP_norm
        if col in invert_cols:
            temp_df[col_name] = 1 - temp_df[col_name]
            
        norm_cols.append(col_name)
    
    # 5. Calcolo Score Finale del Pilastro
    temp_df[risk_name] = temp_df[norm_cols].mean(axis=1)
    
    return temp_df

# --- PILASTRO 1: ECONOMICO ---
# Variabili: Povertà (+), Debito (+), GDP (-)
cols_p1 = ['Population_Below_Poverty_Line_percent', 'Public_Debt_percent_of_GDP', 'Real_GDP_per_Capita_USD']
invert_p1 = ['Real_GDP_per_Capita_USD'] # Alto GDP = Basso Rischio

df_economic = process_pillar(df_risk_pillars, cols_p1, 'Economic Risk', invert_p1)

# --- PILASTRO 2: SOCIALE ---
# Variabili: Mortalità Infantile (+), Disoccupazione Giovani (+), Migrazione Netta (-)
# Nota su Migrazione: Se il tasso è alto (positivo) entra gente (attrattività/stabilità). 
# Se è basso (negativo) gente scappa. Quindi Invertiamo la migrazione.
cols_p2 = ['Infant_Mortality_Rate_x', 'Youth_Unemployment_Rate_y', 'Net_Migration_Rate_x']
invert_p2 = ['Net_Migration_Rate_x']

df_social = process_pillar(df_risk_pillars, cols_p2, 'Social Risk', invert_p2)

# --- PILASTRO 3: INFRASTRUTTURALE ---
# Variabili (Usiamo le DERIVATE): Accesso Elettricità (-), Densità Strade (-), Internet % (-)
# Qui TUTTE vanno invertite perché avere infrastrutture è bene (Basso Rischio)
cols_p3 = ['electricity_access_percent', 'Road_Density', 'Internet_Penetration']
invert_p3 = ['electricity_access_percent', 'Road_Density', 'Internet_Penetration']

df_infrastructure = process_pillar(df_risk_pillars, cols_p3, 'Infrastructure Risk', invert_p3)

# --- PILASTRO 4: DEMOGRAFICO ---
# Variabili: Crescita Pop (+), Età Mediana (-)
# Nota: Bassa età mediana (molti bambini) è un rischio di dipendenza. Alta età = Basso Rischio.
# Nota: Ho escluso Total_Population dal calcolo del rischio (usata solo per Internet %). 
# Avere tanta popolazione non è di per sé un rischio strutturale se l'economia regge.
cols_p4 = ['Population_Growth_Rate_y', 'Median_Age_y']
invert_p4 = ['Median_Age_y'] 

df_demographic = process_pillar(df_risk_pillars, cols_p4, 'Demographic Risk', invert_p4)



In [77]:

# =============================================================================
# 4. UNIONE FINALE (MERGE)
# =============================================================================

# Partiamo dalla lista di tutti i paesi originali per non perderne nessuno
df_final = df_risk_pillars[['Country']].copy()

# Uniamo i 4 dataframe dei rischi
# Usiamo 'left' merge: se un paese non ha dati per un rischio, avrà NaN in quella colonna
df_final = df_final.merge(df_economic[['Country', 'Economic Risk']], on='Country', how='left')
df_final = df_final.merge(df_social[['Country', 'Social Risk']], on='Country', how='left')
df_final = df_final.merge(df_infrastructure[['Country', 'Infrastructure Risk']], on='Country', how='left')
df_final = df_final.merge(df_demographic[['Country', 'Demographic Risk']], on='Country', how='left')

# Calcolo Vulnerabilità Totale (Media dei rischi disponibili)
risk_cols = ['Economic Risk', 'Social Risk', 'Infrastructure Risk', 'Demographic Risk']
df_final['Total Vulnerability'] = df_final[risk_cols].mean(axis=1)

# Pulizia Finale: Per la visualizzazione (Mappa), riempiamo i NaN finali con 0 o lasciamo NaN?
# Solitamente per choropleth map è meglio lasciare NaN (o grigio) per "No Data".
# Ma se vuoi forzare un valore per il codice Dash, puoi usare .fillna(0) qui sotto.
# df_final = df_final.fillna(0)

# Aggiungiamo le coordinate (Lat/Lon) se presenti nel dataset originale 'geo' per il plotting
# (Opzionale, se serve per mapbox, altrimenti Plotly usa i nomi country)

print("Processo completato.")
print(f"Dataset Economico (righe valide): {len(df_economic)}")
print(f"Dataset Sociale (righe valide): {len(df_social)}")
print(f"Dataset Infrastrutture (righe valide): {len(df_infrastructure)}")
print(f"Dataset Demografico (righe valide): {len(df_demographic)}")
print("\nPrime righe del DF Finale:")
print(df_final.head())

# Salvataggio
df_final.to_csv('processed_risk_data.csv', index=False)

Processo completato.
Dataset Economico (righe valide): 259
Dataset Sociale (righe valide): 259
Dataset Infrastrutture (righe valide): 259
Dataset Demografico (righe valide): 259

Prime righe del DF Finale:
          Country  Economic Risk  Social Risk  Infrastructure Risk  \
0     AFGHANISTAN       0.559871     0.730493             0.677311   
1        AKROTIRI            NaN          NaN                  NaN   
2         ALBANIA       0.457675     0.437361             0.666276   
3         ALGERIA       0.360025     0.499839             0.669951   
4  AMERICAN SAMOA       0.477693     0.232493             0.812663   

   Demographic Risk  Total Vulnerability  
0          0.618957             0.646658  
1               NaN                  NaN  
2          0.274719             0.459008  
3          0.425729             0.488886  
4          0.483441             0.501572  


PCA computation

In [78]:
risk_cols = [
    'Economic Risk',
    'Social Risk',
    'Infrastructure Risk',
    'Demographic Risk'
]

X = df_final[risk_cols].values
X_scaled = StandardScaler().fit_transform(X)

# Extract data
X = df_final[risk_cols]

# IMPUTE missing values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# STANDARDIZE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# PCA
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# Store results back into dataframe
df_final['PC1'] = components[:, 0]
df_final['PC2'] = components[:, 1]

Save PCA diagnostics (for report)

In [79]:
explained_variance = pca.explained_variance_ratio_

loadings = pd.DataFrame(
    pca.components_.T,
    index=risk_cols,
    columns=['PC1', 'PC2']
)

display(explained_variance)
display(loadings)

array([0.48707527, 0.18653701])

,PC1,PC2
Economic Risk,0.526478,0.074627
Social Risk,0.476555,0.771571
Infrastructure Risk,0.504635,-0.236555
Demographic Risk,0.490979,-0.585791


Save updated dataset

In [80]:
df_final.to_csv("processed_risk_data.csv", index=False)